In [1]:
from openai import OpenAI
import gradio as gr
import os
import json
import requests

from dotenv import load_dotenv
from services import drug_lookup

In [2]:
mistral_model = "mistral:latest"
llama_model = "llama3.2:1b"
tiny_model = "tinyllama:latest"
neural_model = "neural-chat:latest"

curr_model = mistral_model

# Defining openAI client for local ollama models

# client = OpenAI(
#     base_url="http://localhost:11434/v1",
#     api_key="ollama"  # required but can be any string
# )

# Defining openAI client for OpenAI hosted models
load_dotenv()  # Load environment variables from .env file

gpt_model = "gpt-4o-mini"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# curr_model = llama_model
curr_model = gpt_model

In [3]:
system_prompt = """
You are a helpful pharmaceutical information assistant. Your role is to help users find accurate and reliable information about medications, including their brand names, generic names, active ingredients, purposes, indications, and important safety warnings.

When a user asks about a drug:
1. Use the drug_lookup function to retrieve accurate information from the OpenFDA database
2. Present the information in a clear, organized manner
3. Always emphasize important warnings and safety information
4. If a drug is not found, suggest alternative search terms or recommend consulting a healthcare professional
5. Never provide medical advice—only factual information from the database
6. If information is missing or marked as "N/A", acknowledge this limitation
7. Remind users to consult with healthcare professionals before taking any medication

Always prioritize accuracy and user safety in your responses.
"""

In [4]:
aspririn_info = drug_lookup("Aspirin")
print(json.dumps(aspririn_info, indent=2))

Inside the drug_lookup function
Querying OpenFDA API for brand name: Aspirin
Successfully retrieved data from OpenFDA API
{
  "brand_name": "Low Dose Aspirin",
  "generic_name": "ASPIRIN",
  "active_ingredients": [
    "Active ingredient (in each tablet) Aspirin 81 mg (NSAID)* *nonsteroidal anti-inflammatory drug"
  ],
  "purpose": "Purpose Pain reliever",
  "indications": "Uses for the temporary relief of minor aches and pains or as recommended by your doctor. Because of its delayed action, this product will not provide fast relief of headaches or other symptoms needing immediate relief. ask your doctor about other uses for safety coated 81 mg aspirin",
  "warnings": "Warnings Reye's syndrome : Children and teenagers who have or are recovering from chicken pox or flu-like symptoms should not use this product. When using this product, if changes in behavior with nausea and vomiting occur, consult a doctor because these symptoms could be an early sign of Reye's syndrome, a rare but seri

In [5]:
drug_lookup_llm_schema = {
    "name": "drug_lookup",
    "description": "Query the drug label information for a given drug name.",
    "parameters": {
        "type": "object",
        "properties": {
            "brand_name": {
                "type": "string",
                "description": "The name of the drug to look up."
            }
        },
        "required": ["brand_name"]
    }
}

In [6]:
tools = [{"type": "function", "function": drug_lookup_llm_schema}]
tools

[{'type': 'function',
  'function': {'name': 'drug_lookup',
   'description': 'Query the drug label information for a given drug name.',
   'parameters': {'type': 'object',
    'properties': {'brand_name': {'type': 'string',
      'description': 'The name of the drug to look up.'}},
    'required': ['brand_name']}}}]

In [7]:
def handle_tool_calls(message):
    
    print("Inside handle_tool_calls function")
    print("Received message:", message)

    responses = []

    for tool_call in message.tool_calls:
        print("Processing tool call:", tool_call)

        drug_info = None

        if tool_call.function.name == "drug_lookup":
            try:
                print("Parsing tool call arguments:", tool_call.function.arguments)
                args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError as e:
                print("Error parsing tool call arguments:", e)
                args = {}

            brand_name = args.get("brand_name")

            if not brand_name:
                drug_info = {"error": "No brand name provided."}
            else:   
                print(f"Looking up drug information for: {brand_name}")
                drug_info = drug_lookup(brand_name)
                
        else:
            print(f"Unknown tool called: {tool_call.function.name}")
            drug_info = {"error": f"Unknown tool called: {tool_call.function.name}"}

        response = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(drug_info)
            }

        responses.append(response)
    return responses
            

In [8]:
def med_tool_chat(message, history):

    print("Inside the med_tool_chat function")
    messages = [{"role": "system", "content": system_prompt}]
    
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = content[0]["text"] if content else ""
        messages.append({
            "role": h["role"], 
            "content": content
            })
    
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=curr_model,
        messages=messages,
        temperature=0,
        tools=tools
    )

    max_iterations = 5
    iteration = 0

    while response.choices[0].message.tool_calls and iteration < max_iterations:
        iteration += 1

        assistant_message = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",
            "tool_calls": assistant_message.tool_calls

        })

        tool_responses = handle_tool_calls(assistant_message)
        messages.extend(tool_responses)

        # for msg in messages:
        #     print(f"{msg['role']}: {msg['content']}")

        response = client.chat.completions.create(
            model=curr_model,
            messages=messages,
            temperature=0,
            tools=tools
        )
    
    if iteration == max_iterations and response.choices[0].message.tool_calls:
        print("Max iterations reached while processing tool calls. Some tool calls may not have been handled.") 

    return response.choices[0].message.content or ""

In [ ]:
gr.ChatInterface(fn=med_tool_chat).launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Inside the med_tool_chat function
Inside the med_tool_chat function
Inside handle_tool_calls function
Received message: ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cU7ulfEicBUN5pWhB0KsTre0', function=Function(arguments='{"brand_name":"Tylenol"}', name='drug_lookup'), type='function')])
Processing tool call: ChatCompletionMessageFunctionToolCall(id='call_cU7ulfEicBUN5pWhB0KsTre0', function=Function(arguments='{"brand_name":"Tylenol"}', name='drug_lookup'), type='function')
Parsing tool call arguments: {"brand_name":"Tylenol"}
Looking up drug information for: Tylenol
Inside the drug_lookup function
Querying OpenFDA API for brand name: Tylenol
Successfully retrieved data from OpenFDA API
